# Student Performance EDA · County Education Office
**Purpose:** identify which learners are most at risk of poor performance and which factors are associated with better results, to guide where a learner-support programme should be targeted.

**Data:** `student_performance.csv` (1,200 students, generated with seed 55 by `generate_data.py`). **Structure of this notebook:** 1 Structure · 2 Univariate · 3 Bivariate · 4 Multivariate · 5 Hypotheses, followed by Findings, Recommendations and Limitations.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option("display.width", 200)
pd.options.display.float_format = "{:,.2f}".format
plt.rcParams["figure.dpi"] = 60          # small inline previews; saved files in figures/ stay at 150 dpi
students = pd.read_csv("student_performance.csv")

## 1. Structure
**Grain:** one row per student, 1,200 rows, no date column. Here is what each column is, and how it is treated in the analysis.

In [2]:
students.head()

,student_id,county,school_type,gender,attendance_rate,study_hours,math,english,science
0,1,Turkana,Private,M,0.77,6.50,81.00,71.00,73.00
1,2,Nyeri,Private,M,0.97,8.50,100.00,86.00,90.00
2,3,Nyeri,Private,F,0.98,8.00,82.00,89.00,90.00
3,4,Turkana,Public,M,0.93,6.00,83.00,91.00,72.00
4,5,Kisumu,Private,F,0.67,6.10,76.00,69.00,62.00


In [3]:
print(students.shape)
students.info()

(1200, 9)
<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   student_id       1200 non-null   int64  
 1   county           1200 non-null   str    
 2   school_type      1200 non-null   str    
 3   gender           1200 non-null   str    
 4   attendance_rate  1200 non-null   float64
 5   study_hours      1200 non-null   float64
 6   math             1200 non-null   float64
 7   english          1200 non-null   float64
 8   science          1200 non-null   float64
dtypes: float64(5), int64(1), str(3)
memory usage: 84.5 KB


| Column | Variable type | Notes |
|---|---|---|
| `student_id` | Identifier (nominal) | A label, not a quantity; excluded from every statistic. |
| `county` | Categorical, nominal (6 values) | Nairobi, Kisumu, Nakuru, Mombasa, Nyeri, Turkana. |
| `school_type` | Categorical, nominal (2) | Public / Private. |
| `gender` | Categorical, nominal (2) | F / M. |
| `attendance_rate` | Numerical, continuous (proportion 0–1) | Share of school days attended. |
| `study_hours` | Numerical, continuous (hours, assumed per week) | Right-skewed; lead with the median. |
| `math`, `english`, `science` | Numerical, continuous (score 0–100) | Subject scores. |

In [4]:
checks = pd.Series({
    "rows": len(students),
    "missing values (all columns)": int(students.isna().sum().sum()),
    "duplicate rows": int(students.duplicated().sum()),
    "duplicate student_id": int(students["student_id"].duplicated().sum()),
    "attendance_rate outside 0-1": int(((students["attendance_rate"] < 0) | (students["attendance_rate"] > 1)).sum()),
    "study_hours below 0": int((students["study_hours"] < 0).sum()),
    "subject scores outside 0-100": int(((students[["math", "english", "science"]] < 0) | (students[["math", "english", "science"]] > 100)).sum().sum()),
    "subject scores exactly 100": int((students[["math", "english", "science"]] == 100).sum().sum()),
})
print(checks.to_string())
for col in ["county", "school_type", "gender"]:
    print(col, sorted(students[col].unique()))

rows                            1200
missing values (all columns)       0
duplicate rows                     0
duplicate student_id               0
attendance_rate outside 0-1        0
study_hours below 0                0
subject scores outside 0-100       0
subject scores exactly 100        48
county ['Kisumu', 'Mombasa', 'Nairobi', 'Nakuru', 'Nyeri', 'Turkana']
school_type ['Private', 'Public']
gender ['F', 'M']


**Data-quality conclusion.** There are no missing values, no duplicate rows and no duplicate IDs; every attendance rate is a valid proportion and every score is within 0–100. The categorical columns contain only the expected labels. One note: 48 subject scores are exactly 100, the ceiling, which suggests scores were capped at 100 (so the very top of the scale is slightly compressed). **The data are clean enough to analyse without repair.**

### 1.2 Derived columns
`average_score` is the mean of the three subject scores. `performance_band` uses `pd.cut`: **At Risk** below 60, **Meeting** 60 to below 75, **Exceeding** 75 and above.

In [5]:
score_cols = ["math", "english", "science"]
students["average_score"] = students[score_cols].mean(axis=1)
students["performance_band"] = pd.cut(students["average_score"], bins=[0, 60, 75, 100.01], right=False,
                                      labels=["At Risk", "Meeting", "Exceeding"])
print("unbanded rows:", students["performance_band"].isna().sum())
students[["student_id", "math", "english", "science", "average_score", "performance_band"]].head()

unbanded rows: 0


,student_id,math,english,science,average_score,performance_band
0,1,81.00,71.00,73.00,75.00,Exceeding
1,2,100.00,86.00,90.00,92.00,Exceeding
2,3,82.00,89.00,90.00,87.00,Exceeding
3,4,83.00,91.00,72.00,82.00,Exceeding
4,5,76.00,69.00,62.00,69.00,Meeting


In [6]:
print(students.groupby("performance_band", observed=True)["average_score"].agg(["min", "max", "count"]))

                   min    max  count
performance_band                    
At Risk          48.33  59.67     99
Meeting          60.00  74.67    672
Exceeding        75.00 100.00    429


Each band's minimum and maximum respect the cut-offs (At Risk below 60, Meeting from 60 to below 75, Exceeding from 75), so the binning is correct.

## 2. Univariate analysis

In [7]:
numeric = ["attendance_rate", "study_hours", "math", "english", "science", "average_score"]
summary = pd.DataFrame({
    "mean": students[numeric].mean(),
    "median": students[numeric].median(),
    "std": students[numeric].std(),
    "IQR": students[numeric].quantile(0.75) - students[numeric].quantile(0.25),
    "skewness": students[numeric].skew(),
    "min": students[numeric].min(),
    "max": students[numeric].max(),
})
summary

,mean,median,std,IQR,skewness,min,max
attendance_rate,0.80,0.82,0.12,0.17,-0.86,0.33,1.00
study_hours,5.08,4.50,3.03,3.80,1.17,0.20,23.30
math,71.89,72.00,10.86,14.00,0.10,37.00,100.00
english,74.52,74.00,10.49,14.00,0.16,47.00,100.00
science,69.61,69.00,12.02,17.00,0.11,35.00,100.00
average_score,72.01,71.67,9.11,12.00,0.26,48.33,100.00


**Reading the table.**
- **Centre and spread:** the average score has mean 72.0, median 71.7 and standard deviation 9.1, so mean and median agree and most students sit within about 9 points of 72. English is the strongest subject (mean 74.5) and science the weakest and most variable (mean 69.6, standard deviation 12.0).
- **Attendance** is high and left-skewed (skewness -0.86): the median is 0.82, above the mean of 0.80, because a minority of students attend far less than the rest.
- **Study hours** are right-skewed (skewness 1.17): the median is 4.5 hours but the mean is 5.1, pulled up by a few heavy studiers. For this variable the **median is the better description of a typical student**.

In [8]:
for col in ["county", "school_type", "gender", "performance_band"]:
    out = pd.DataFrame({"count": students[col].value_counts(), "share_pct": students[col].value_counts(normalize=True) * 100})
    print(f"--- {col} ---"); print(out.round(1).to_string(), "\n")

--- county ---
         count  share_pct
county                   
Mombasa    212      17.70
Kisumu     208      17.30
Nyeri      203      16.90
Nakuru     202      16.80
Turkana    191      15.90
Nairobi    184      15.30 

--- school_type ---
             count  share_pct
school_type                  
Public         866      72.20
Private        334      27.80 

--- gender ---
        count  share_pct
gender                  
F         612      51.00
M         588      49.00 

--- performance_band ---
                  count  share_pct
performance_band                  
Meeting             672      56.00
Exceeding           429      35.80
At Risk              99       8.20 



**Categorical mix.** The six counties are close to equal in size (184 to 212 students each). 72% of students are in public schools and 28% in private schools; gender is balanced (51.0% F). By performance band, **8.2% of students (99) are At Risk**, 56.0% are Meeting and 35.8% are Exceeding.

### 2.2 Outliers in `study_hours` (IQR method) and the histogram

In [9]:
q1, q3 = students["study_hours"].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = students[students["study_hours"] > upper]
print(f"Q1={q1:.1f}  Q3={q3:.1f}  IQR={iqr:.1f}  fences: {lower:.1f} to {upper:.1f}")
print("students above the upper fence:", len(outliers), f"({len(outliers) / len(students) * 100:.1f}%)  | max = {students['study_hours'].max()}")
print("students below the lower fence:", int((students["study_hours"] < lower).sum()))

Q1=2.9  Q3=6.7  IQR=3.8  fences: -2.8 to 12.4
students above the upper fence: 29 (2.4%)  | max = 23.3
students below the lower fence: 0


In [10]:
ax = students["study_hours"].plot(kind="hist", bins=30, color="#9a9a9a", edgecolor="white", figsize=(8, 4))
ax.set_title("Most students study 2 to 7 hours a week; a long tail reaches 23 hours")
ax.set_xlabel("Study hours per week"); ax.set_ylabel("Number of students")
ax.axvline(students["study_hours"].median(), color="#8E0000", linestyle="--", label=f"Median {students['study_hours'].median():.1f} h")
ax.legend(); plt.tight_layout()
plt.savefig("figures/eda_study_hours_hist.png", dpi=150); plt.close()   # shown below via the saved file

![eda_study_hours_hist](figures/eda_study_hours_hist.png)

**Shape and outliers.** The histogram (`figures/eda_study_hours_hist.png`) is unimodal and **right-skewed**: most students study between 2.9 and 6.7 hours (the middle half) and a long tail runs to 23.3 hours. The IQR rule puts the upper fence at 12.4 hours and flags **29 students (2.4%)**; none fall below the lower fence. These are high but not impossible values (about 3 hours a day), and they are consistent with the smooth tail of the distribution, so they are **kept and flagged, not deleted**. Because they are real students who may well be high achievers, removing them would bias the study-time analysis.

## 3. Bivariate analysis

In [11]:
corr_pearson = students[numeric].corr(method="pearson")
corr_spearman = students[numeric].corr(method="spearman")
print("Pearson:"); print(corr_pearson.round(2)); print("\nSpearman:"); print(corr_spearman.round(2))

Pearson:
                 attendance_rate  study_hours  math  english  science  average_score
attendance_rate             1.00        -0.03  0.30     0.33     0.26           0.36
study_hours                -0.03         1.00  0.56     0.62     0.55           0.70
math                        0.30         0.56  1.00     0.56     0.46           0.81
english                     0.33         0.62  0.56     1.00     0.50           0.83
science                     0.26         0.55  0.46     0.50     1.00           0.82
average_score               0.36         0.70  0.81     0.83     0.82           1.00

Spearman:
                 attendance_rate  study_hours  math  english  science  average_score
attendance_rate             1.00        -0.03  0.30     0.31     0.24           0.34
study_hours                -0.03         1.00  0.52     0.58     0.51           0.66
math                        0.30         0.52  1.00     0.53     0.44           0.79
english                     0.31         0.58

In [12]:
fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(corr_pearson, annot=True, fmt=".2f", cmap="Greys", vmin=0, vmax=1, linewidths=0.5, ax=ax)
ax.set_title("Study hours and the three subject scores move together (Pearson correlation)")
plt.tight_layout(); plt.savefig("figures/eda_correlation_heatmap.png", dpi=150); plt.close()   # shown below via the saved file

![eda_correlation_heatmap](figures/eda_correlation_heatmap.png)

In [13]:
students["is_private"] = (students["school_type"] == "Private").astype(int)
with_avg = students[numeric + ["is_private"]].corr()["average_score"].drop("average_score").sort_values(ascending=False)
print(with_avg.round(2))

english           0.83
science           0.82
math              0.81
study_hours       0.70
attendance_rate   0.36
is_private        0.27
Name: average_score, dtype: float64


**Interpretation (correlation with `average_score`).**
1. **The three subject scores (r = 0.83, 0.82, 0.81)** are the strongest correlations, but this is *mechanical*: `average_score` is built from them, so it says nothing about causes. Excluding them, the real drivers are:
2. **`study_hours` (r = 0.70, Spearman 0.66)**: strong and positive. Students who study more score higher; study time alone accounts for about half of the variation in scores (R² = 0.50).
3. **`attendance_rate` (r = 0.36)**: moderate and positive. Better attendance goes with higher scores, but the link is much weaker than for study hours.
4. **School type (private = 1; r = 0.27)**: a modest positive link; private-school students score higher on average (examined in section 4).

Pearson and Spearman agree closely, so the relationships are not driven by a few extreme values. These are associations, and they do not prove that study time or attendance *causes* higher scores.

### 3.2 Average scores by gender, school type and county

In [14]:
def group_table(col):
    return (students.groupby(col, observed=True)
            .agg(students=("student_id", "size"), mean_score=("average_score", "mean"), median_score=("average_score", "median"),
                 at_risk_pct=("performance_band", lambda x: (x == "At Risk").mean() * 100))
            .sort_values("mean_score", ascending=False))
for col in ["gender", "school_type", "county"]:
    print(f"--- {col} ---"); print(group_table(col).round(1), "\n")

--- gender ---
        students  mean_score  median_score  at_risk_pct
gender                                                 
F            612       72.10         71.70         9.30
M            588       71.90         71.30         7.10 

--- school_type ---
             students  mean_score  median_score  at_risk_pct
school_type                                                 
Private           334       76.00         75.30         1.50
Public            866       70.50         69.70        10.90 

--- county ---
         students  mean_score  median_score  at_risk_pct
county                                                  
Kisumu        208       73.10         72.50         8.20
Mombasa       212       72.20         72.20         9.00
Nairobi       184       72.00         72.20         9.20
Nyeri         203       71.90         71.30         8.90
Nakuru        202       71.80         70.80         6.40
Turkana       191       71.00         70.00         7.90 



In [15]:
county = group_table("county").sort_values("at_risk_pct", ascending=False)
county["95%_margin_pts"] = 1.96 * np.sqrt(county["at_risk_pct"] / 100 * (1 - county["at_risk_pct"] / 100) / county["students"]) * 100
print(county[["students", "at_risk_pct", "95%_margin_pts"]].round(1))
chi2, p_county, _, _ = stats.chi2_contingency(pd.crosstab(students["county"], students["performance_band"] == "At Risk"))
print(f"\nchi-square test, county vs At Risk: p = {p_county:.2f}")
t, p_gender = stats.ttest_ind(students.loc[students["gender"] == "F", "average_score"], students.loc[students["gender"] == "M", "average_score"])
print(f"t-test, gender gap in average score: p = {p_gender:.2f}")

         students  at_risk_pct  95%_margin_pts
county                                        
Nairobi       184         9.20            4.20
Mombasa       212         9.00            3.80
Nyeri         203         8.90            3.90
Kisumu        208         8.20            3.70
Turkana       191         7.90            3.80
Nakuru        202         6.40            3.40

chi-square test, county vs At Risk: p = 0.92
t-test, gender gap in average score: p = 0.73


In [16]:
ax = county["at_risk_pct"].plot(kind="bar", color="#9a9a9a", figsize=(8, 4), rot=0)
ax.set_title("At Risk share by county: highest in Nairobi, but the counties are close")
ax.set_ylabel("Students At Risk (%)"); ax.set_xlabel("")
for p in ax.patches: ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom")
plt.tight_layout(); plt.savefig("figures/eda_at_risk_by_county.png", dpi=150); plt.close()   # shown below via the saved file

![eda_at_risk_by_county](figures/eda_at_risk_by_county.png)

In [17]:
ax = students.boxplot(column="average_score", by="school_type", figsize=(6, 4.5), grid=False)
plt.suptitle(""); ax.set_title("Private-school students score higher on average")
ax.set_xlabel(""); ax.set_ylabel("Average score (0-100)")
plt.tight_layout(); plt.savefig("figures/eda_score_by_school_type_box.png", dpi=150); plt.close()   # shown below via the saved file

![eda_score_by_school_type_box](figures/eda_score_by_school_type_box.png)

**Findings.**
- **School type:** private-school students average 76.0 against 70.5 for public, a gap of 5.5 points; and only 1.5% of private-school students are At Risk against 10.9% of public-school students (about 7× higher).
- **Gender:** female and male students average 72.1 and 71.9, a difference of 0.2 points (p = 0.73), so there is **no meaningful gender gap**.
- **County:** **Nairobi has the highest At Risk share (9.2%)** and Nakuru the lowest (6.4%). With only 184 to 212 students per county the margin of error on each share is roughly ±4 points, and a chi-square test finds the differences statistically indistinguishable from chance (p = 0.92). So **county is not a reliable basis for targeting**; the figures are shown in `figures/eda_at_risk_by_county.png` and `figures/eda_score_by_school_type_box.png`.

## 4. Multivariate analysis
Is the private-school advantage real, or is it because private-school students attend more or study more? First, do the two groups differ on the other drivers?

In [18]:
print(students.groupby("school_type")[["attendance_rate", "study_hours"]].mean().round(2))

             attendance_rate  study_hours
school_type                              
Private                 0.80         5.00
Public                  0.80         5.11


In [19]:
students["attendance_band"] = pd.cut(students["attendance_rate"], bins=[0, 0.70, 0.85, 1.0], labels=["Below 70%", "70-85%", "Above 85%"])
students["study_quartile"] = pd.qcut(students["study_hours"], 4, labels=["Q1", "Q2", "Q3", "Q4"])

by_attendance = students.pivot_table(index="attendance_band", columns="school_type", values="average_score", aggfunc="mean", observed=True)
by_attendance["gap (Private - Public)"] = by_attendance["Private"] - by_attendance["Public"]
by_study = students.pivot_table(index="study_quartile", columns="school_type", values="average_score", aggfunc="mean", observed=True)
by_study["gap (Private - Public)"] = by_study["Private"] - by_study["Public"]
print("Mean average_score by attendance band and school type"); print(by_attendance.round(1))
print("\nMean average_score by study-hour quartile and school type"); print(by_study.round(1))

Mean average_score by attendance band and school type
school_type      Private  Public  gap (Private - Public)
attendance_band                                         
Below 70%          71.50   64.90                    6.60
70-85%             75.50   69.80                    5.70
Above 85%          79.00   74.00                    5.00

Mean average_score by study-hour quartile and school type
school_type     Private  Public  gap (Private - Public)
study_quartile                                         
Q1                69.50   63.80                    5.70
Q2                74.90   67.10                    7.80
Q3                76.20   71.20                    5.00
Q4                83.60   80.10                    3.60


In [20]:
print("Students per cell (attendance band x school type):")
print(students.pivot_table(index="attendance_band", columns="school_type", values="student_id", aggfunc="count", observed=True))
print("\nStudents per cell (study quartile x school type):")
print(students.pivot_table(index="study_quartile", columns="school_type", values="student_id", aggfunc="count", observed=True))
print("\nStudy-hour quartile cut-offs:", students["study_hours"].quantile([0.25, 0.5, 0.75]).round(1).tolist())

Students per cell (attendance band x school type):
school_type      Private  Public
attendance_band                 
Below 70%             71     167
70-85%               134     365
Above 85%            129     334

Students per cell (study quartile x school type):
school_type     Private  Public
study_quartile                 
Q1                   84     227
Q2                   86     216
Q3                   82     206
Q4                   82     217

Study-hour quartile cut-offs: [2.9, 4.5, 6.7]


**Interpretation.** Private and public students are almost identical on attendance (0.80 vs 0.80) and study hours (5.0 vs 5.1), so neither can explain the gap. Comparing like with like, **the private-school advantage persists in every group**: between 5.0 and 6.6 points across the three attendance bands and between 3.6 and 7.8 points across the four study-hour quartiles. It is somewhat smaller among the heaviest studiers (Q4, 3.6 points), where public-school students also score 80.1 on average, possibly because scores are capped at 100, which limits how far anyone can score above the rest. The gap is therefore **associated with school type itself, not with attendance or study time**. It cannot be attributed to school quality alone: household income, teacher ratios and prior attainment are not in this dataset.

### 4.3 Trend line: average score against study hours
The dataset has no dates, so the trend is fitted across students: `np.polyfit` of `average_score` on `study_hours`.

In [21]:
slope, intercept = np.polyfit(students["study_hours"], students["average_score"], 1)
r_squared = np.corrcoef(students["study_hours"], students["average_score"])[0, 1] ** 2
print(f"slope = {slope:.2f} points per study hour | intercept = {intercept:.1f} | R-squared = {r_squared:.2f}")
print(f"range of study_hours used for the fit: {students['study_hours'].min()} to {students['study_hours'].max()}")
print(f"prediction at 40 hours (outside the data): {intercept + slope * 40:.0f}")

ax = students.plot(kind="scatter", x="study_hours", y="average_score", alpha=0.3, color="#9a9a9a", figsize=(8, 5))
xs = np.linspace(students["study_hours"].min(), students["study_hours"].max(), 100)
ax.plot(xs, intercept + slope * xs, color="#8E0000", linewidth=2.5, label=f"Trend: +{slope:.1f} points per hour (R² = {r_squared:.2f})")
ax.set_title("Each extra weekly study hour goes with about 2 more points")
ax.set_xlabel("Study hours per week"); ax.set_ylabel("Average score (0-100)"); ax.legend()
plt.tight_layout(); plt.savefig("figures/eda_study_hours_trend.png", dpi=150); plt.close()   # shown below via the saved file

slope = 2.12 points per study hour | intercept = 61.3 | R-squared = 0.50
range of study_hours used for the fit: 0.2 to 23.3
prediction at 40 hours (outside the data): 146


![eda_study_hours_trend](figures/eda_study_hours_trend.png)

**What the slope means.** On average, each additional hour of weekly study is associated with about **2.1 more points** on the average score (R² = 0.50, so study hours explain about half of the variation between students).

**Limitation for prediction.** The line is straight and was fitted on 0.2 to 23.3 hours, with most students under 11; extending it is unsafe: the points visibly flatten above about 12 hours and the line itself passes 100 at about 18 hours (it would predict a score of 146 for 40 hours, which is impossible on a 0–100 scale), and a correlation of this kind does not prove that extra study causes the improvement (see `figures/eda_study_hours_trend.png`).

## 5. Hypotheses
The EDA suggests three hypotheses, each tested against the data below, and one thing the data cannot tell us.

In [22]:
risk = students["performance_band"] == "At Risk"
h = pd.DataFrame({
    "hypothesis": ["H1: more study time is associated with higher scores",
                   "H2: low attendance is associated with a higher chance of being At Risk",
                   "H3: the private-school gap is not explained by attendance or study time"],
    "evidence": [f"slope {slope:.2f} points/hour, R2 {r_squared:.2f}; At Risk {students.loc[students['study_quartile']=='Q1','performance_band'].eq('At Risk').mean()*100:.1f}% in Q1 vs {students.loc[students['study_quartile']=='Q4','performance_band'].eq('At Risk').mean()*100:.1f}% in Q4",
                 f"At Risk {students.loc[students['attendance_band']=='Below 70%','performance_band'].eq('At Risk').mean()*100:.1f}% below 70% attendance vs {students.loc[students['attendance_band']=='Above 85%','performance_band'].eq('At Risk').mean()*100:.1f}% above 85%",
                 f"gap persists in all {len(by_attendance) + len(by_study)} attendance/study groups (min {pd.concat([by_attendance.iloc[:, 2], by_study.iloc[:, 2]]).min():.1f} points)"],
})
h["verdict"] = ["Supported", "Supported", "Supported (association only)"]
h

,hypothesis,evidence,verdict
0,H1: more study time is associated with higher ...,"slope 2.12 points/hour, R2 0.50; At Risk 21.2%...",Supported
1,H2: low attendance is associated with a higher...,At Risk 23.9% below 70% attendance vs 2.6% abo...,Supported
2,H3: the private-school gap is not explained by...,gap persists in all 7 attendance/study groups ...,Supported (association only)


In [23]:
print((students.groupby("study_quartile", observed=True)["performance_band"].apply(lambda x: (x == "At Risk").mean() * 100)).round(1).rename("At Risk % by study quartile"))
print()
print((students.groupby("attendance_band", observed=True)["performance_band"].apply(lambda x: (x == "At Risk").mean() * 100)).round(1).rename("At Risk % by attendance band"))
print()
print(students.groupby("performance_band", observed=True)[["attendance_rate", "study_hours"]].mean().round(2))

study_quartile
Q1   21.20
Q2    7.90
Q3    2.80
Q4    0.30
Name: At Risk % by study quartile, dtype: float64

attendance_band
Below 70%   23.90
70-85%       6.00
Above 85%    2.60
Name: At Risk % by attendance band, dtype: float64

                  attendance_rate  study_hours
performance_band                              
At Risk                      0.68         2.45
Meeting                      0.80         4.00
Exceeding                    0.84         7.38


- **H1 supported:** every extra hour goes with about 2.1 more points, and the At Risk share falls from 21.2% in the lowest study quartile to 0.3% in the highest.
- **H2 supported:** At Risk share is 23.9% when attendance is below 70% and 2.6% above 85%. At Risk students attend 68% of days and study 2.5 hours, against 84% and 7.4 hours for Exceeding students.
- **H3 supported, as an association:** the private-school gap stays at 3.6 to 7.8 points in every comparison group.
- **What the data cannot tell us:** *why* private-school students score higher, and whether study time or attendance would *cause* better results if increased. Household income, teacher-to-student ratios, class size and prior attainment are not in the dataset, and without them school type may be standing in for those factors.

## Findings
1. **8.2% of students (99 of 1,200) are At Risk** (average score below 60); 56.0% are Meeting and 35.8% are Exceeding. The typical student scores about 72.
2. **Study time is the strongest lever:** each extra weekly study hour is associated with about 2.1 more points (r = 0.70, R² = 0.50). The At Risk share is 21.2% in the lowest study quartile (up to 2.9 hours) and 0.3% in the highest.
3. **Attendance matters, less strongly:** r = 0.36; the At Risk share is 23.9% below 70% attendance against 2.6% above 85%.
4. **Public-school students are 7× as likely to be At Risk** (10.9% vs 1.5%). The 5.5-point score gap holds within every attendance band and study-hour quartile, so it is not explained by those two factors.
5. **County and gender do not separate learners:** the At Risk share ranges only from 6.4% (Nakuru) to 9.2% (Nairobi), a difference consistent with chance (p = 0.92), and male and female averages differ by 0.2 points.

## Recommendations
1. **Target by behaviour, not by county or gender.** Enrol learners who study 2.9 hours a week or less (the lowest quartile) or attend fewer than 70% of days. This rule covers 39% of learners and captures **87% of all At Risk students**; the At Risk rate is 18.4% inside the group against 1.8% outside. Offer supervised study time and attendance follow-up as the core of the programme.
2. **Give public schools priority in the first phase, and measure why.** Their At Risk rate is about 7× higher. Because the data cannot say whether the cause is teaching, resources or home circumstances, collect household income, teacher ratios and prior-year results, and run the programme as a pilot with a comparison group before scaling.

## Limitations
This is an observational, cross-sectional dataset: all findings are **associations, not causes**, and nothing here proves that raising study hours or attendance would raise scores. Important drivers (household income, teacher quality, prior attainment, health, distance to school) are missing, so school type may be standing in for them. The data are synthetic and come from one point in time, so we cannot see whether learners are improving or declining. Subject scores appear capped at 100, which compresses the top of the scale; how study hours were measured is unknown (the unit is assumed to be per week); and the 29 high study-hours values were kept. County samples of about 200 students leave wide margins of error, so county differences should not drive decisions. The trend line is linear and valid only within the observed 0.2 to 23 hours.